# NumPy 最终练习题（含提示、可运行答案与自动检测）
每题结构: 题目（Markdown） -> 提示（可折叠） -> 答案代码（可运行） -> 自动检测代码（运行后给出通过信息或 assertion 错误）。
请依次运行每对单元来完成自测。


### 题 1

数据类型与转换（编程）

创建一个包含整数 `1, 2, 3` 的 NumPy 数组，显式指定为 `np.int32`。将其转换为 `np.float64`，并说明 dtype 变化与内存影响（简述）。


<details><summary>提示（点击展开）</summary>

提示：使用 dtype 参数创建数组，用 astype 转换类型；比较 .dtype 和 itemsize。

</details>


In [ ]:
# 答案代码：Q1
import numpy as np
arr = np.array([1,2,3], dtype=np.int32)
print('orig dtype, itemsize:', arr.dtype, arr.itemsize)
arr2 = arr.astype(np.float64)
print('converted dtype, itemsize:', arr2.dtype, arr2.itemsize)
# 保存 variables for autograder
_q1_arr = arr
_q1_arr2 = arr2


In [ ]:
# 自动检测 Q1
assert _q1_arr.dtype == np.int32, "原数组 dtype 应为 int32"
assert _q1_arr2.dtype == np.float64, "转换后 dtype 应为 float64"
assert _q1_arr2.itemsize == 8, "float64 每元素应占 8 字节"
print('Q1 OK')

### 题 2

创建与特殊矩阵（编程）

用一行代码创建一个 6x6 的矩阵，其对角线为 1，主对角线下方第一条次对角线为 2，主对角线上方第一条次对角线为 3，其余元素为 0。


<details><summary>提示（点击展开）</summary>

提示：可以先用 zeros 再用 np.fill_diagonal，也可以用 np.diag 和拼接。

</details>


In [ ]:
# 答案代码：Q2
import numpy as np
M = np.zeros((6,6), dtype=int)
np.fill_diagonal(M, 1)
i = np.arange(5)
M[i+1, i] = 2
M[i, i+1] = 3
print(M)
_q2_M = M


In [ ]:
# 自动检测 Q2
assert _q2_M.shape == (6,6)
# check diagonals
assert all(_q2_M.diagonal() == 1)
assert all(np.diag(_q2_M, k=-1) == 2)
assert all(np.diag(_q2_M, k=1) == 3)
print('Q2 OK')

### 题 3

reshape 与 -1（编程）

有一个一维数组 `np.arange(30)`。将其重塑为形状 `(2, 3, -1)`，并说明第三维为何自动计算出多少，以及总元素数如何匹配。


<details><summary>提示（点击展开）</summary>

提示：使用 reshape(2,3,-1)，并查看 .shape 属性。

</details>


In [ ]:
# 答案代码：Q3
import numpy as np
arr = np.arange(30)
mat = arr.reshape(2,3,-1)
print('shape:', mat.shape)
_q3_shape = mat.shape


In [ ]:
# 自动检测 Q3
assert _q3_shape == (2,3,5), f"期望形状 (2,3,5)，但得到 {_q3_shape}"
print('Q3 OK')

### 题 4

视图与拷贝（简答+编程）

给出例子证明：`reshape` 在何种情况下返回视图（view），何种情况返回拷贝（copy）。编写代码创建一个原数组，执行 `reshape`，并用 `arr.base` 或 `np.may_share_memory` 验证。


<details><summary>提示（点击展开）</summary>

提示：对连续内存（C-contiguous）的数组 reshape 通常返回 view；对非连续切片 reshape 可能返回 copy。

</details>


In [ ]:
# 答案代码：Q4
import numpy as np
a = np.arange(12)          # C-contiguous
b = a.reshape(3,4)         # view expected
print('b.base is a?', b.base is a)

c = a[::2]                 # non-contiguous view (stride != 1)
print('c.base is a?', c.base is a)
# reshape c to shape that requires copying sometimes
try:
    d = c.reshape(2,3)
    print('d.base is c?', d.base is c)
except Exception as e:
    d = c.copy()
    print('reshape required copy, made copy')

# use may_share_memory
from numpy import may_share_memory
_share_b = may_share_memory(a,b)
_share_d = may_share_memory(a,d)
print('may_share b with a:', _share_b)
print('may_share d with a:', _share_d)
_q4 = dict(a=a, b=b, c=c, d=d, share_b=_share_b, share_d=_share_d)


In [ ]:
# 自动检测 Q4
assert _q4['b'].base is _q4['a'] or may_share_memory(_q4['a'], _q4['b'])
# d might be copy depending on reshape; ensure memory may not share for copy case
print('Q4 OK')

### 题 5

广播规则（编程）

给定 `A.shape = (4,1,6)` 和 `B.shape = (3,6)`，在 NumPy 中尝试 `A + B`。请说明会怎样进行广播（包括中间将 B 视为何形状），并给出能使这两者可相加的代码示例。


<details><summary>提示（点击展开）</summary>

提示：在左侧补 1 使 B 变为 (1,3,6)，然后广播到 (4,3,6)。可以用 B[np.newaxis,:,:]。

</details>


In [ ]:
# 答案代码：Q5
import numpy as np
A = np.zeros((4,1,6))
B = np.arange(18).reshape(3,6)
try:
    C = A + B
    print('A+B shape', C.shape)
except Exception as e:
    print('direct A+B failed:', e)

C2 = A + B[np.newaxis, :, :]
print('A + B[np.newaxis,:,:] shape:', C2.shape)
_q5_shape = C2.shape


In [ ]:
# 自动检测 Q5
assert _q5_shape == (4,3,6)
print('Q5 OK')

### 题 6

高级索引（编程）

给定数组 `X = np.arange(24).reshape(4,6)`，使用 fancy indexing（整数数组索引）选出第 0、2 行以及第 1、4 列交叉的元素，形成一个 2x2 的数组。


<details><summary>提示（点击展开）</summary>

提示：使用 np.ix_ 构造行列索引网格。

</details>


In [ ]:
# 答案代码：Q6
import numpy as np
X = np.arange(24).reshape(4,6)
rows = [0,2]
cols = [1,4]
result = X[np.ix_(rows, cols)]
print(result)
_q6 = result


In [ ]:
# 自动检测 Q6
import numpy as np
assert isinstance(_q6, np.ndarray)
assert _q6.shape == (2,2)
assert list(_q6.flatten()) == [1,4,13,16]
print('Q6 OK')

### 题 7

布尔掩码与 np.where（编程）

生成 `arr = np.random.randint(-10, 10, size=15)`，用布尔掩码将所有负数替换为其绝对值（正数），但保留 0 与正数不变。要求使用 **两种不同方法**：一种用布尔索引，另一种用 `np.where`。


<details><summary>提示（点击展开）</summary>

提示：arr[arr<0] = -arr[arr<0] ； np.where(arr<0, -arr, arr)

</details>


In [ ]:
# 答案代码：Q7
import numpy as np
np.random.seed(0)
arr = np.random.randint(-10, 10, size=15)
arr1 = arr.copy()
mask = arr1 < 0
arr1[mask] = -arr1[mask]

arr2 = np.where(arr < 0, -arr, arr)
print('orig:', arr)
print('method1:', arr1)
print('method2:', arr2)
_q7_orig = arr; _q7_m1 = arr1; _q7_m2 = arr2


In [ ]:
# 自动检测 Q7
import numpy as np
assert np.all(_q7_m1 >= 0)
assert np.all(_q7_m2 >= 0)
assert np.array_equal(_q7_m1, _q7_m2)
print('Q7 OK')

### 题 8

ufunc 与 reduce（编程）

用 `np.add.reduce` 与 `np.add.accumulate` 分别作用于 `np.array([1,2,3,4])`，并说明两者的区别与典型应用场景。


<details><summary>提示（点击展开）</summary>

提示：reduce 返回单个累计值，accumulate 返回每步的累计数组。

</details>


In [ ]:
# 答案代码：Q8
import numpy as np
a = np.array([1,2,3,4])
r = np.add.reduce(a)
acc = np.add.accumulate(a)
print('reduce:', r)
print('accumulate:', acc)
_q8_r = r; _q8_acc = acc


In [ ]:
# 自动检测 Q8
import numpy as np
assert _q8_r == 10
assert np.array_equal(_q8_acc, np.array([1,3,6,10]))
print('Q8 OK')

### 题 9

矩阵乘法与 einsum（编程）

给定 `A.shape = (2,3)` 与 `B.shape = (3,4)`，用三种方法计算矩阵乘积 `C = A @ B`：`@` 运算符、`np.dot`/`np.matmul`、以及 `np.einsum`。说明 `einsum` 的字符串表达及其一般优势。


<details><summary>提示（点击展开）</summary>

提示：'ik,kj->ij'

</details>


In [ ]:
# 答案代码：Q9
import numpy as np
A = np.arange(6).reshape(2,3)
B = np.arange(12).reshape(3,4)
C1 = A @ B
C2 = np.dot(A,B)
C3 = np.matmul(A,B)
C4 = np.einsum('ik,kj->ij', A, B)
print('equal?', np.allclose(C1, C2) and np.allclose(C1, C4))
_q9_C1=C1; _q9_C4=C4


In [ ]:
# 自动检测 Q9
import numpy as np
assert np.array_equal(_q9_C1, _q9_C4)
print('Q9 OK')

### 题 10

线性代数：求解与条件数（编程+简答）

构造一个接近奇异的 3x3 矩阵（例如使用某一行接近另一行的方式），用 `np.linalg.solve` 解线性系统 `Ax = b`（任选 b），并计算其条件数 `np.linalg.cond(A)`。解释条件数大说明什么，并演示通过改变矩阵（稍微增加扰动）导致解的不稳定性。


<details><summary>提示（点击展开）</summary>

提示：构造具有近线性相关行的矩阵，计算 cond，比较解在 b 改变时的差异。

</details>


In [ ]:
# 答案代码：Q10
import numpy as np
A = np.array([[1,2,3],[2,4.0001,6],[1,0.9999,2]], dtype=float)
b = np.array([1,2,3.], dtype=float)
condA = np.linalg.cond(A)
x = np.linalg.solve(A,b)
# perturb b slightly
b2 = b + 1e-6*np.array([1, -1, 0.5])
x2 = np.linalg.solve(A, b2)
print('cond:', condA)
print('x:', x)
print('x2:', x2)
_q10 = dict(A=A,b=b,b2=b2,cond=condA,x=x,x2=x2)


In [ ]:
# 自动检测 Q10
import numpy as np
assert _q10['cond'] > 1e3  or True  # just show cond exists
# check that small perturbation leads to some difference
diff = np.linalg.norm(_q10['x2'] - _q10['x'])
print('perturbation caused diff:', diff)
print('Q10 OK')

### 题 11

特征值与 SVD（编程）

随机生成一个 5x5 的浮点矩阵 `M`（用种子以保证可重复），计算并比较 `np.linalg.eig(M)` 与 `np.linalg.svd(M)` 的输出（讨论何时用 eig，何时用 SVD）。


<details><summary>提示（点击展开）</summary>

提示：eig 返回 eigenvalues/vectors（方阵），SVD 对任意矩阵适用并返回奇异值。

</details>


In [ ]:
# 答案代码：Q11
import numpy as np
np.random.seed(0)
M = np.random.randn(5,5)
eigvals, eigvecs = np.linalg.eig(M)
U,s,Vt = np.linalg.svd(M)
print('eigvals shape', eigvals.shape)
print('s shape', s.shape)
_q11 = dict(M=M,eigvals=eigvals,s=s)


In [ ]:
# 自动检测 Q11
import numpy as np
assert _q11['eigvals'].shape == (5,)
assert _q11['s'].shape == (5,)
print('Q11 OK')

### 题 12

广播与内存开销（偏难，编程）

使用 `np.ones((1000, 1000))` 与一个形状 `(1000,)` 的向量 `v`。衡量并比较 `A + v`（广播得到的临时数组）与 `A + v[np.newaxis, :]` 两种写法的内存与时间开销（可用 `%timeit` 或 `time` 测量），说明广播不复制时与何时会产生临时数组。


<details><summary>提示（点击展开）</summary>

提示：两种写法等价，都会产生输出数组；展示时间差异并说明 out= 可以避免临时。

</details>


In [ ]:
# 答案代码：Q12
import numpy as np, time
A = np.ones((1000,1000))
v = np.arange(1000.0)
t0 = time.time()
C = A + v
t1 = time.time()
C2 = A + v[np.newaxis, :]
t2 = time.time()
print('A+v time:', t1-t0)
print('A+v[np.newaxis,:] time:', t2-t1)
# show in-place to avoid temp
A_copy = A.copy()
t3 = time.time()
A_copy += v  # in-place uses broadcasting to match shape but writes into A_copy
t4 = time.time()
print('in-place A_copy += v time:', t4-t3)
_q12 = dict(t_ab=t1-t0, t_ab2=t2-t1, t_inplace=t4-t3)


In [ ]:
# 自动检测 Q12
assert 't_ab' in _q12
print('Q12 OK')

### 题 13

缺失值与统计（编程）

创建一个含有 NaN 的数组 `arr = np.array([1.0, np.nan, 2.0, 3.0, np.nan])`。计算元素的均值与中位数，要求忽略 NaN，分别用 `np.nanmean`/`np.nanmedian` 与掩码实现两种方法，并比较结果。


<details><summary>提示（点击展开）</summary>

提示：使用 np.isnan 掩码。

</details>


In [ ]:
# 答案代码：Q13
import numpy as np
arr = np.array([1.0, np.nan, 2.0, 3.0, np.nan])
m1 = np.nanmean(arr)
med1 = np.nanmedian(arr)
mask = ~np.isnan(arr)
m2 = arr[mask].mean()
med2 = np.median(arr[mask])
print('nanmean, nanmedian', m1, med1)
print('mask mean, median', m2, med2)
_q13 = dict(m1=m1, med1=med1, m2=m2, med2=med2)


In [ ]:
# 自动检测 Q13
import numpy as np
assert abs(_q13['m1'] - _q13['m2']) < 1e-12
assert abs(_q13['med1'] - _q13['med2']) < 1e-12
print('Q13 OK')

### 题 14

排序与 argsort（编程）

给定二维数组 `scores`（形状 `(N, M)`）表示 N 个学生 M 门课成绩，使用 `np.argsort` 找出总分排名前 3 的学生索引，并返回他们的总分与成绩行。请给出示例数据并实现。


<details><summary>提示（点击展开）</summary>

提示：先 sum(axis=1)，再 argsort，选取前 3。

</details>


In [ ]:
# 答案代码：Q14
import numpy as np
np.random.seed(0)
scores = np.random.randint(50,100, size=(10,4))
total = scores.sum(axis=1)
top3_idx = np.argsort(-total)[:3]
top3_totals = total[top3_idx]
top3_rows = scores[top3_idx]
print('top3_idx', top3_idx)
print('totals', top3_totals)
_q14 = dict(idx=top3_idx, totals=top3_totals, rows=top3_rows)


In [ ]:
# 自动检测 Q14
assert len(_q14['idx']) == 3
assert _q14['totals'].shape[0] == 3
print('Q14 OK')

### 题 15

文件 I/O（编程）

将一个随机生成的 2D 浮点数组保存为 `.npz`（使用 `np.savez_compressed`），然后读取回来并验证内容相同。同时展示如何用 `np.savetxt` 保存为 CSV 并用 `np.loadtxt` 读取（注意格式问题）。


<details><summary>提示（点击展开）</summary>

提示：使用 np.allclose 验证数值相等。

</details>


In [ ]:
# 答案代码：Q15
import numpy as np, os
np.random.seed(0)
arr = np.random.randn(5,5)
np.savez_compressed('demo_test.npz', arr=arr)
data = np.load('demo_test.npz')
arr2 = data['arr']
assert np.allclose(arr, arr2)
np.savetxt('demo_test.csv', arr, delimiter=',', fmt='%.6f')
arr3 = np.loadtxt('demo_test.csv', delimiter=',')
assert np.allclose(arr, arr3)
print('saved and loaded OK')
_q15 = dict(arr=arr, arr2=arr2, arr3=arr3, files=['demo_test.npz','demo_test.csv'])


In [ ]:
# 自动检测 Q15
import numpy as np, os
assert np.allclose(_q15['arr'], _q15['arr2'])
assert np.allclose(_q15['arr'], _q15['arr3'])
print('Q15 OK')

### 题 16

结构化数组（编程）

创建一个结构化数组（structured array），包含字段 `name`（长度 10 的字符串）、`age`（int）、`score`（float）。插入 4 条记录，然后按 `score` 排序返回排序后的名字列表。


<details><summary>提示（点击展开）</summary>

提示：使用 np.dtype 定义字段，使用 argsort 按 score 排序。

</details>


In [ ]:
# 答案代码：Q16
import numpy as np
dt = np.dtype([('name','U10'), ('age', 'i4'), ('score', 'f8')])
data = np.array([('Alice',23,88.5), ('Bob',19,92.0), ('Cathy',22,78.0), ('David',21,85.0)], dtype=dt)
idx = np.argsort(data['score'])[::-1]
sorted_names = data['name'][idx]
print(sorted_names)
_q16 = sorted_names


In [ ]:
# 自动检测 Q16
assert list(_q16)[0] == 'Bob'
assert len(_q16) == 4
print('Q16 OK')

### 题 17

stride 与内存视图（偏难，编程+解释）

创建 `a = np.arange(16).reshape(4,4)`，然后使用切片创建 `b = a[:, ::2]`（取每行的偶数列）。打印 `a.strides` 和 `b.strides`，解释 stride 的含义，并说明为什么 `b` 是一个视图而非拷贝。


<details><summary>提示（点击展开）</summary>

提示：strides 单位是字节，表示沿每个轴移动一步所需字节数，b.base 指向 a。

</details>


In [ ]:
# 答案代码：Q17
import numpy as np
a = np.arange(16).reshape(4,4)
b = a[:, ::2]
print('a.strides:', a.strides)
print('b.strides:', b.strides)
print('b.base is a?', b.base is a)
_q17 = dict(a=a, b=b, strides_a=a.strides, strides_b=b.strides, base_is_a=(b.base is a))


In [ ]:
# 自动检测 Q17
assert _q17['base_is_a'] is True
print('Q17 OK')

### 题 18

性能优化：向量化 vs Python 循环（编程/测量）

给定一个大数组 `x = np.random.rand(10_000_00)`（一百万元素），实现以下两种运算并比较时间：
- 使用 Python 循环逐元素计算 `y[i] = x[i]**2 + 2*x[i] + 1`。
- 使用 NumPy 向量化表达式 `y = x**2 + 2*x + 1`。
报告两者时间差并简短解释原因。


<details><summary>提示（点击展开）</summary>

提示：为节省时间，测试规模可设置为 200,000 或 100,000；向量化明显快很多。

</details>


In [ ]:
# 答案代码：Q18
import numpy as np, time
np.random.seed(0)
n = 200_000
x = np.random.rand(n)
t0 = time.time()
y_loop = np.empty_like(x)
for i in range(n):
    y_loop[i] = x[i]**2 + 2*x[i] + 1
t1 = time.time()
t_loop = t1-t0
t2 = time.time()
y_vec = x**2 + 2*x + 1
t3 = time.time()
t_vec = t3-t2
print('loop time:', t_loop)
print('vectorized time:', t_vec)
print('speedup:', t_loop / t_vec if t_vec>0 else float('inf'))
_q18 = dict(t_loop=t_loop, t_vec=t_vec)


In [ ]:
# 自动检测 Q18
assert _q18['t_vec'] < _q18['t_loop']
print('Q18 OK, vectorized faster')

### 题 19

广播陷阱（偏难，编程）

设计一个例子展示广播可能导致不期望的结果（例如当一个维度为 1 的数组意外扩展时导致计算错误），并说明如何修改代码以避免错误（例如使用 `np.expand_dims` 或重塑 `reshape`）。


<details><summary>提示（点击展开）</summary>

提示：示例可用列向量 vs 行向量相加的方向错误。

</details>


In [ ]:
# 答案代码：Q19
import numpy as np
col = np.array([[1],[2],[3]])   # shape (3,1)
row_wrong = np.array([[10,20,30]])  # shape (1,3) is fine, but wrong use could be (3,)
# Mistake: using row as (3,) might broadcast unexpectedly if you intended column-wise op
row_flat = np.array([10,20,30])  # shape (3,)
# If you mistakenly do col + row_flat.reshape(3,1) you get shape (3,1) which is different
bad = col + row_flat.reshape(3,1)  # results in (3,1) broadcasting along wrong axis if intended (3,3)
good = col + row_flat[np.newaxis, :]  # correct (3,3)
print('bad shape', bad.shape, 'good shape', good.shape)
_q19 = dict(bad=bad, good=good)


In [ ]:
# 自动检测 Q19
assert _q19['good'].shape == (3,3)
print('Q19 OK')

### 题 20

综合题：实现一个小功能（编程）

编写一个函数 `moving_average(arr, k)`，使用 NumPy 的 `convolve` 或 `cumsum` 高效计算一维数组 `arr` 的窗口大小为 `k` 的移动平均（边界处理为“有效”模式，即输出长度为 `len(arr)-k+1`）。要求使用向量化实现并给出时间复杂度简要分析。


<details><summary>提示（点击展开）</summary>

提示：使用 cumsum，注意前缀和技巧。

</details>


In [ ]:
# 答案代码：Q20
import numpy as np
def moving_average(arr, k):
    arr = np.asarray(arr, dtype=float)
    if k <= 0 or k > arr.size:
        raise ValueError("k must be between 1 and len(arr)")
    c = np.cumsum(arr)
    # prefix trick: sum of window i..i+k-1 = c[i+k-1] - c[i-1] (with c[-1]=0 handling)
    out = (c[k-1:] - np.concatenate(([0.0], c[:-k]))) / k
    return out

# example
x = np.arange(10)
print('moving_average:', moving_average(x, 3))
_q20 = moving_average(x, 3)


In [ ]:
# 自动检测 Q20
import numpy as np
assert np.allclose(_q20, np.array([1.,2.,3.,4.,5.,6.,7.,8.]))
print('Q20 OK')